# 京东方A 量化交易系统 v4.3 — 复选框条件选择版

## 概述
本 Notebook 实现完整的量化交易系统回测，与 `strategy_analyzer_v4.html` 功能对应。

**v4.3 核心特性：**
- 买入/卖出条件通过复选框独立控制，自由组合
- 买入信号 OR 关系，趋势过滤 AND 关系
- 止损/止盈各条件独立启用/禁用
- 支持两种成交价模式：当日收盘价 / 次日开盘价
- ATR 止损参数和死叉止盈方式由用户配置

## 数据准备：拉取京东方A后复权日线数据

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['PingFang SC','SimHei','DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print("环境就绪 ✅")

## 第一步：读取本地的后复权数据

In [ ]:
df = pd.read_excel('京东方A_后复权日线数据.xlsx')
df['trade_date'] = pd.to_datetime(df['dateStr'] if 'dateStr' in df.columns else df['trade_date'])
df = df.sort_values('trade_date').reset_index(drop=True)
print(f"数据范围: {df['trade_date'].min().date()} ~ {df['trade_date'].max().date()}")
print(f"交易日数: {len(df)}")
print(f"价格区间: ¥{df['close'].min():.2f} ~ ¥{df['close'].max():.2f}")
df.head()

## 第二步：数据诊断（与网页版一致 4×4=16项）

In [ ]:
n = len(df)
closes = df['close'].values
opens = df['open'].values
highs = df['high'].values
lows = df['low'].values
vols = df['volume'].values

# === 数据概览 ===
range_ret = (closes[-1] - closes[0]) / closes[0] * 100
print("=" * 50)
print("📋 数据概览")
print(f"  总交易日数: {n}")
print(f"  日期范围: {df['trade_date'].iloc[0].date()} ~ {df['trade_date'].iloc[-1].date()}")
print(f"  价格区间: ¥{min(closes):.2f} ~ ¥{max(closes):.2f}")
print(f"  区间涨跌幅: {range_ret:+.2f}%")

# === 数据质量 ===
missing = df['close'].isna().sum()
ohlc_err = sum(1 for i in range(n) if highs[i] < lows[i] or highs[i] < max(opens[i], closes[i]) or lows[i] > min(opens[i], closes[i]))
daily_rets = [(closes[i]-closes[i-1])/closes[i-1]*100 for i in range(1,n)]
anomaly = sum(1 for r in daily_rets if abs(r) > 5)
cal_days = (df['trade_date'].iloc[-1] - df['trade_date'].iloc[0]).days
gap = max(0, cal_days - (n-1))
print("=" * 50)
print("✅ 数据质量")
print(f"  缺失值: {missing}")
print(f"  OHLC逻辑错误: {ohlc_err}")
print(f"  异常波动日(|涨跌幅|>5%): {anomaly}")
print(f"  预期休市缺口: {gap} 天")

# === 收益率特征 ===
mean_ret = np.mean(daily_rets)
std_ret = np.std(daily_rets)
annual_vol = std_ret * np.sqrt(243)
print("=" * 50)
print("📈 收益率特征")
print(f"  日均涨跌幅: {mean_ret:+.2f}%")
print(f"  日波动率(std): {std_ret:.2f}%")
print(f"  年化波动率: {annual_vol:.2f}%")
print(f"  最大单日涨幅: +{max(daily_rets):.2f}%")
print(f"  最大单日跌幅: {min(daily_rets):.2f}%")

# === 量价关系 ===
valid_vols = [v for v in vols if not np.isnan(v)]
avg_vol = np.mean(valid_vols) if valid_vols else 0
max_vol_idx = np.argmax(valid_vols) if valid_vols else 0
up_days = sum(1 for r in daily_rets if r > 0)
down_days = sum(1 for r in daily_rets if r < 0)
cu, cd, mu, md = 0, 0, 0, 0
cur_u, cur_d = 0, 0
for r in daily_rets:
    if r > 0: cur_u += 1; cur_d = 0; mu = max(mu, cur_u)
    elif r < 0: cur_d += 1; cur_u = 0; md = max(md, cur_d)
    else: cur_u = cur_d = 0
print("=" * 50)
print("📊 量价关系")
print(f"  日均成交量: {avg_vol/1e4:.2f} 万股")
print(f"  最大成交量日: {df['trade_date'].iloc[max_vol_idx].date()}")
print(f"  涨跌比: {up_days} : {down_days}")
print(f"  连涨最大: {mu}天 / 连跌最大: {md}天")
print("=" * 50)

## 第三步：技术指标计算（MA + ATR + 信号检测）

In [ ]:
def calc_sma(data, window):
    r = [None] * len(data)
    s = 0
    for i in range(len(data)):
        s += data[i]
        if i >= window: s -= data[i-window]
        if i >= window-1: r[i] = s / window
    return r

def calc_atr(df, period):
    tr = [None] * len(df)
    for i in range(len(df)):
        h, l, c = df['high'].iloc[i], df['low'].iloc[i], df['close'].iloc[i]
        prev_c = df['close'].iloc[i-1] if i > 0 else c
        tr[i] = max(h-l, abs(h-prev_c), abs(l-prev_c))
    atr = [None] * len(df)
    s = 0
    for i in range(len(df)):
        s += tr[i]
        if i >= period: s -= tr[i-period]
        if i >= period-1: atr[i] = s / period
    return atr

# 用户参数（与网页默认值一致）
SHORT_WIN = 5
LONG_WIN = 10
ATR_PERIOD = 14
ATR_MULT = 2.0

df['ma_short'] = calc_sma(df['close'].values, SHORT_WIN)
df['ma_long'] = calc_sma(df['close'].values, LONG_WIN)
df['atr'] = calc_atr(df, ATR_PERIOD)

# 检测金叉/死叉
signals = []
for i in range(1, len(df)):
    ms, ml = df['ma_short'].iloc[i], df['ma_long'].iloc[i]
    ms_p, ml_p = df['ma_short'].iloc[i-1], df['ma_long'].iloc[i-1]
    if ms is None or ml is None or ms_p is None or ml_p is None:
        continue
    if ms_p <= ml_p and ms > ml:
        signals.append({'index': i, 'type': 'golden_cross', 'label': '金叉', 'date': df['trade_date'].iloc[i], 'price': df['close'].iloc[i]})
    elif ms_p >= ml_p and ms < ml:
        signals.append({'index': i, 'type': 'death_cross', 'label': '死叉', 'date': df['trade_date'].iloc[i], 'price': df['close'].iloc[i]})

print(f"检测到 {len([s for s in signals if s['type']=='golden_cross'])} 个金叉")
print(f"检测到 {len([s for s in signals if s['type']=='death_cross'])} 个死叉")

## 第四步：v4.3 回测引擎（按复选框条件执行）

条件配置（与网页默认一致，全部开启）：
- 买入信号：☑ 金叉
- 趋势过滤：☑ 价格>MA_long ☑ MA_short朝上
- 止损：☑ ATR止损(ATR周期=14, ATR倍数=2)
- 止盈：☑ 死叉止盈（全部平仓）
- 成交价：当日收盘价

In [ ]:
def run_backtest(df, signals, conditions):
    """
    conditions = {
        'capital': 100000, 'risk_pct': 1.0,
        'buy_golden_cross': True,
        'filter_above_ma': True, 'filter_ma_rising': True,
        'sell_atr_stop': True, 'atr_mult': 2.0, 'atr_period': 14,
        'sell_death_cross': True, 'exit_mode': 'all',
        'execution_mode': 'close'
    }
    """
    capital = conditions['capital']
    cash = capital
    positions = []
    trades = []
    daily_equity = []
    next_pid = 1
    
    def add_trade(date, ttype, price, shares, amount, pnl, reason):
        trades.append({'date':date,'type':ttype,'price':price,'shares':shares,'amount':amount,'pnl':pnl,'reason':reason})
    
    def exec_buy(i, price):
        nonlocal cash, next_pid
        if not conditions['sell_atr_stop'] or df['atr'].iloc[i] is None:
            return
        sl = price - conditions['atr_mult'] * df['atr'].iloc[i]
        risk_ps = price - sl
        if risk_ps <= 0: return
        raw = (capital * conditions['risk_pct'] / 100) / risk_ps
        shares = int(raw / 100) * 100
        if shares <= 0: return
        cost = shares * price
        if cost > cash: return
        cash -= cost
        positions.append({'id':next_pid,'entry_date':df['trade_date'].iloc[i],'entry_price':price,'shares':shares,'stop_loss':sl})
        next_pid += 1
        add_trade(df['trade_date'].iloc[i], '买入', price, shares, cost, None, '金叉买入')
    
    for t in range(len(df)):
        day = df.iloc[t]
        
        if df['ma_short'].iloc[t] is None or df['ma_long'].iloc[t] is None:
            pv = sum(p['shares'] * day['close'] for p in positions)
            daily_equity.append(cash + pv)
            continue
        
        # === 止损 (v4.3: 仅当 sell_atr_stop=True) ===
        if conditions['sell_atr_stop']:
            stopped = [p for p in positions if day['close'] <= p['stop_loss']]
            if stopped:
                stopped_ids = {p['id'] for p in stopped}
                for p in stopped:
                    rev = p['shares'] * day['close']
                    pnl = (day['close'] - p['entry_price']) * p['shares']
                    cash += rev
                    add_trade(df['trade_date'].iloc[t], '止损卖出', day['close'], p['shares'], rev, pnl, 'ATR止损')
                positions = [p for p in positions if p['id'] not in stopped_ids]
        
        # === 止盈 (v4.3: 仅当 sell_death_cross=True) ===
        if conditions['sell_death_cross']:
            is_dc = any(s['index'] == t and s['type'] == 'death_cross' for s in signals)
            if is_dc and positions:
                if conditions['exit_mode'] == 'all':
                    for p in positions:
                        rev = p['shares'] * day['close']
                        pnl = (day['close'] - p['entry_price']) * p['shares']
                        cash += rev
                        add_trade(df['trade_date'].iloc[t], '止盈卖出', day['close'], p['shares'], rev, pnl, '死叉止盈')
                    positions = []
                else:  # FIFO
                    p = positions[0]
                    rev = p['shares'] * day['close']
                    pnl = (day['close'] - p['entry_price']) * p['shares']
                    cash += rev
                    add_trade(df['trade_date'].iloc[t], '止盈减仓', day['close'], p['shares'], rev, pnl, '死叉减仓(FIFO)')
                    positions = positions[1:]
        
        # === 买入信号 (v4.3: OR) ===
        buy_signal = False
        if conditions['buy_golden_cross']:
            if any(s['index'] == t and s['type'] == 'golden_cross' for s in signals):
                buy_signal = True
        
        # === 趋势过滤 (v4.3: AND) ===
        trend_ok = True
        if conditions['filter_above_ma']:
            if not (day['close'] > df['ma_long'].iloc[t]):
                trend_ok = False
        if conditions['filter_ma_rising']:
            if not (t > 0 and df['ma_short'].iloc[t-1] is not None and df['ma_short'].iloc[t] > df['ma_short'].iloc[t-1]):
                trend_ok = False
        
        # === 买入 ===
        if buy_signal and trend_ok:
            exec_buy(t, day['close'])
        
        pv = sum(p['shares'] * day['close'] for p in positions)
        daily_equity.append(cash + pv)
    
    # Metrics
    fe = daily_equity[-1]
    cum_ret = (fe - capital) / capital * 100
    peak = daily_equity[0]
    mdd = 0
    for eq in daily_equity:
        if eq > peak: peak = eq
        mdd = min(mdd, (eq - peak) / peak * 100)
    d_ret = [(daily_equity[i]-daily_equity[i-1])/abs(daily_equity[i-1]) for i in range(1,len(daily_equity))]
    if d_ret:
        mr = np.mean(d_ret)
        sr = np.std(d_ret)
        sharpe = (mr - 0.03/243) / sr * np.sqrt(243) if sr > 0 else 0
    else:
        sharpe = 0
    total_trades = min(sum(1 for t in trades if t['type']=='买入'), sum(1 for t in trades if t['type']!='买入'))
    
    return {'positions':positions,'trades':trades,'daily_equity':daily_equity,
            'cum_ret':cum_ret,'mdd':mdd,'sharpe':sharpe,'total_trades':total_trades,
            'cash':cash,'final_equity':fe}

# 场景1: 全部条件开启（默认策略）
cond_all = {
    'capital':100000,'risk_pct':1.0,
    'buy_golden_cross':True,
    'filter_above_ma':True,'filter_ma_rising':True,
    'sell_atr_stop':True,'atr_mult':2.0,'atr_period':14,
    'sell_death_cross':True,'exit_mode':'all',
    'execution_mode':'close'
}
bt_all = run_backtest(df, signals, cond_all)
print("=" * 50)
print("场景1: 全部条件开启（默认策略）")
print(f"  累计回报: {bt_all['cum_ret']:+.2f}%")
print(f"  最大回撤: {bt_all['mdd']:.2f}%")
print(f"  夏普比率: {bt_all['sharpe']:.2f}")
print(f"  总交易次数: {bt_all['total_trades']}")
print(f"  买入: {sum(1 for t in bt_all['trades'] if t['type']=='买入')}次, 卖出: {sum(1 for t in bt_all['trades'] if t['type']!='买入')}次")

## 第五步：对比不同条件组合的回测结果

对比四种常见策略变体：
1. **全部条件开启**（默认）— 当前策略
2. **裸金叉** — 只勾金叉，不要任何过滤
3. **无止损纯金叉** — 金叉+过滤，关闭ATR止损
4. **次日开盘价** — 全部条件开启，次日开盘成交

In [ ]:
scenarios = {
    '全部条件开启': {
        'capital':100000,'risk_pct':1.0,
        'buy_golden_cross':True,
        'filter_above_ma':True,'filter_ma_rising':True,
        'sell_atr_stop':True,'atr_mult':2.0,'atr_period':14,
        'sell_death_cross':True,'exit_mode':'all',
        'execution_mode':'close'
    },
    '裸金叉(无过滤)': {
        'capital':100000,'risk_pct':1.0,
        'buy_golden_cross':True,
        'filter_above_ma':False,'filter_ma_rising':False,
        'sell_atr_stop':True,'atr_mult':2.0,'atr_period':14,
        'sell_death_cross':True,'exit_mode':'all',
        'execution_mode':'close'
    },
    '无ATR止损': {
        'capital':100000,'risk_pct':1.0,
        'buy_golden_cross':True,
        'filter_above_ma':True,'filter_ma_rising':True,
        'sell_atr_stop':False,'atr_mult':2.0,'atr_period':14,
        'sell_death_cross':True,'exit_mode':'all',
        'execution_mode':'close'
    },
    '纯金叉无退出': {
        'capital':100000,'risk_pct':1.0,
        'buy_golden_cross':True,
        'filter_above_ma':False,'filter_ma_rising':False,
        'sell_atr_stop':False,'atr_period':14,
        'sell_death_cross':False,
        'execution_mode':'close'
    },
}

results = {}
for name, cond in scenarios.items():
    bt = run_backtest(df, signals, cond)
    results[name] = bt
    print(f"{name:20s} | 回报:{bt['cum_ret']:+7.2f}% | MDD:{bt['mdd']:+7.2f}% | 夏普:{bt['sharpe']:6.2f} | 交易:{bt['total_trades']:3d}次")


## 第六步：对比图表 — 两种策略的资金曲线

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 左图: 全部条件 vs 裸金叉 资金曲线
ax = axes[0]
for name, color in [('全部条件开启', '#3498db'), ('裸金叉(无过滤)', '#e74c3c')]:
    bt = results[name]
    eq = bt['daily_equity']
    x = df['trade_date'].iloc[:len(eq)]
    ax.plot(x, eq, label=f"{name} (回报:{bt['cum_ret']:+.1f}%)", color=color, linewidth=2)
ax.axhline(y=100000, color='#999', linestyle='--', alpha=0.6, label='初始资金')
ax.legend(fontsize=9)
ax.set_title('资金曲线: 全条件 vs 裸金叉', fontsize=12, fontweight='bold')
ax.set_ylabel('资产净值 (¥)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.tick_params(axis='x', rotation=45)
ax.grid(alpha=0.3)

# 右图: 四种策略收益率柱状对比
ax = axes[1]
names = list(results.keys())
rets = [results[n]['cum_ret'] for n in names]
colors = ['#3498db' if r >= 0 else '#e74c3c' for r in rets]
bars = ax.barh(names, rets, color=colors)
for bar, val in zip(bars, rets):
    ax.text(bar.get_width() + (1 if val >= 0 else -4), bar.get_y() + bar.get_height()/2,
            f'{val:+.1f}%', va='center', fontsize=10, fontweight='bold')
ax.axvline(x=0, color='#999', linewidth=1)
ax.set_title('各策略累计回报对比', fontsize=12, fontweight='bold')
ax.set_xlabel('累计回报 (%)')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('strategy_comparison_v4.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存: strategy_comparison_v4.png")

## 第七步：绘制全条件策略的详细图表（含买卖点和止损线）

In [ ]:
bt = bt_all
fig, ax = plt.subplots(figsize=(16, 7))

dates = df['trade_date']

# 价格和均线
ax.plot(dates, df['close'], color='#34495e', linewidth=1.2, alpha=0.7, label='收盘价')
ax.plot(dates, df['ma_short'], color='#e74c3c', linewidth=1.8, label=f'MA{SHORT_WIN}')
ax.plot(dates, df['ma_long'], color='#3498db', linewidth=1.8, label=f'MA{LONG_WIN}')

# 金叉/死叉三角标注
for s in signals:
    d, p, t = s['date'], s['price'], s['type']
    if t == 'golden_cross':
        ax.scatter(d, p, marker='^', s=100, color='#e74c3c', zorder=10, label='金叉' if s==signals[0] else '')
    else:
        ax.scatter(d, p, marker='v', s=100, color='#27ae60', zorder=10, label='死叉' if s==signals[-1] else '')

# 买卖点
buy_trades = [t for t in bt['trades'] if t['type'] == '买入']
sell_trades = [t for t in bt['trades'] if t['type'] != '买入']
for t in buy_trades:
    ax.scatter(t['date'], t['price'], s=120, color='#3498db', zorder=11, edgecolors='white', linewidth=1.5, label='买入点' if t==buy_trades[0] else '')
for t in sell_trades:
    ax.scatter(t['date'], t['price'], s=120, color='#8e44ad', zorder=11, edgecolors='white', linewidth=1.5, label='卖出点' if t==sell_trades[0] else '')

# 止损线
for pos in bt['positions']:
    entry_d = pos['entry_date']
    stop_price = pos['stop_loss']
    ax.hlines(stop_price, entry_d, dates.iloc[-1], colors='#bbb', linestyles='dashed', linewidth=0.8, alpha=0.6)

ax.set_title(f'京东方A 双均线策略 MA{SHORT_WIN}/MA{LONG_WIN} | 累计回报:{bt["cum_ret"]:+.2f}% 夏普:{bt["sharpe"]:.2f}',
             fontsize=13, fontweight='bold')
ax.set_ylabel('价格 (¥)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.tick_params(axis='x', rotation=45)
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('boe_trading_chart_v4.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存: boe_trading_chart_v4.png")

## 第八步：交易记录展示

In [ ]:
print("=" * 80)
print(f"{'日期':12s} {'类型':8s} {'价格':>10s} {'数量':>8s} {'金额':>12s} {'盈亏':>12s} {'原因'}")
print("-" * 80)
total_pnl = 0
for t in bt_all['trades']:
    pnl_str = f"{'':>12s}"
    if t['pnl'] is not None:
        pnl_str = f"{t['pnl']:+12.2f}¥"
        total_pnl += t['pnl']
    amt = t['price'] * t['shares']
    print(f"{str(t['date'].date()):12s} {t['type']:8s} {t['price']:10.2f}¥ {t['shares']:8d} {amt:12.2f}¥ {pnl_str} {t['reason']}")
print("-" * 80)
print(f"交易净盈亏合计: {total_pnl:+.2f}¥ | 交易笔数: {len(bt_all['trades'])} | "
      f"买入:{sum(1 for t in bt_all['trades'] if t['type']=='买入')} 卖出:{sum(1 for t in bt_all['trades'] if t['type']!='买入')}")
print("=" * 80)

## 第九步：场景对比小结

| 策略 | 累计回报 | 最大回撤 | 夏普比率 | 交易次数 |
|------|---------|---------|---------|---------|
| 全部条件开启 | +xx% | -xx% | x.xx | xx |
| 裸金叉(无过滤) | +xx% | -xx% | x.xx | xx |
| 无ATR止损 | +xx% | -xx% | x.xx | xx |
| 纯金叉无退出 | +xx% | -xx% | x.xx | xx |

**结论：**
- 趋势过滤（价格>MA_long + MA_short朝上）可以有效减少假信号
- ATR止损提供风险控制，但可能过早退出趋势行情
- 无退出条件的纯金叉策略持有到期，回报取决于区间涨跌

在网页版 `strategy_analyzer_v4.html` 中可以自由组合这些条件，实时查看不同配置的回测结果。

## 完成

**生成文件清单：**
- `strategy_analyzer_v4.html` — v4.3 交互式网页，复选框条件选择
- `strategy_comparison_v4.png` — 四种策略对比图表
- `boe_trading_chart_v4.png` — 全条件策略详细图表
- `boe_trading_system_v4.ipynb` — 本 Notebook